# Analysis

Analysis notebook structure
Section 1 — Data quality & basic shape
Null counts, dtypes, value distributions. Make sure the data is trustworthy before drawing any conclusions.
Section 2 — Event type deep dive
Distribution of TRADE/REDEEM/MERGE/REWARD/MAKER_REBATE per wallet. Understanding the activity composition before doing any PnL work.
Section 3 — Resolve logic
Per-market PnL reconstruction. Classify every market each wallet touched as WIN / LOSS / OPEN. This is the foundation for all performance metrics.
Section 4 — Performance metrics
Win rate, total PnL, realized vs unrealized, Sortino, Calmar, cumulative PnL curve over time.
Section 5 — Behavioral analysis
Trading frequency, time-of-day patterns, position sizing, holding periods, market category preferences.
Section 6 — Strategy reverse engineering
What kinds of markets do they bet on, at what implied probabilities, how do they size, do they average in or out, any detectable signals in their entry timing.
Section 7 — Cross-wallet comparison
Do they trade the same markets? Do they copy each other? Who leads, who follows?

## Import

In [2]:
import pandas as pd
import numpy as np
import json
from datetime import datetime, timezone
from collections import Counter

# Disable Arrow-backed strings — same fix as fetch notebook
pd.options.future.infer_string = False

# Display settings — show all columns, readable floats
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", "{:.4f}".format)

WALLETS   = ["rn1", "sovereign", "swisstony"]
DATA_DIR  = "data"

In [3]:
dfs = {}

for name in WALLETS:
    path = f"{DATA_DIR}/activity_{name}.parquet"
    df   = pd.read_parquet(path, engine="fastparquet")
    df["wallet"]   = name                                          # add wallet label
    df["datetime"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)  # human readable time
    dfs[name] = df
    print(f"{name}: {len(df):,} rows | {df['datetime'].min().date()} → {df['datetime'].max().date()}")

# Combined dataframe for cross-wallet analysis
activity = pd.concat(dfs.values(), ignore_index=True)
print(f"\nCombined: {len(activity):,} total rows")
print(f"Columns: {list(activity.columns)}")

rn1: 2,298,316 rows | 2025-07-09 → 2026-04-22
sovereign: 1,271,417 rows | 2025-08-07 → 2026-04-22
swisstony: 3,276,271 rows | 2025-08-09 → 2026-04-22

Combined: 6,846,004 total rows
Columns: ['proxyWallet', 'timestamp', 'conditionId', 'type', 'size', 'usdcSize', 'transactionHash', 'price', 'asset', 'side', 'outcomeIndex', 'title', 'slug', 'icon', 'eventSlug', 'outcome', 'name', 'pseudonym', 'bio', 'profileImage', 'profileImageOptimized', 'wallet', 'datetime']


## Data Quality and Basic Shape

In [6]:
print("=" * 50)
print("DTYPES")
print("=" * 50)
print(activity.dtypes)

print("\n" + "=" * 50)
print("NULL COUNTS (combined)")
print("=" * 50)
nulls = activity.isnull().sum()
nulls_pct = (nulls / len(activity) * 100).round(2)
null_summary = pd.DataFrame({"null_count": nulls, "null_pct": nulls_pct})
print(null_summary[null_summary["null_count"] > 0])

DTYPES
proxyWallet                          object
timestamp                             int64
conditionId                          object
type                                 object
size                                float64
usdcSize                            float64
transactionHash                      object
price                               float64
asset                                object
side                                 object
outcomeIndex                          int64
title                                object
slug                                 object
icon                                 object
eventSlug                            object
outcome                              object
name                                 object
pseudonym                            object
bio                                  object
profileImage                         object
profileImageOptimized                object
wallet                               object
datetime                 